In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import random
import os

# 1. Cố định Seed
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(2026)

print("--- 🔥 BÀI TẬP NÂNG CAO: PHÂN LOẠI ĐA LỚP & ĐÁNH GIÁ VALIDATION ---")

# 2. Tạo dữ liệu giả lập (300 mẫu, 8 đặc trưng, 3 lớp nhãn: 0, 1, 2)
X_raw = np.random.randn(300, 8)
y_raw = np.random.randint(0, 3, size=300)

# Chia tập Train (80%) và Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw, test_size=0.2, random_state=2026, stratify=y_raw)

# 3. Custom Dataset
class MultiClassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) # CrossEntropyLoss yêu cầu kiểu long

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

train_loader = DataLoader(MultiClassDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiClassDataset(X_val, y_val), batch_size=32, shuffle=False)

# 4. Định nghĩa Mô hình Phân loại 3 Lớp
class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MultiClassClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes) # Đầu ra 3 nodes, KHÔNG thêm Softmax
        )
    def forward(self, x):
        return self.net(x)

model = MultiClassClassifier(input_dim=8, num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 5. Training Loop kết hợp Đánh giá Validation
epochs = 15
for epoch in range(1, epochs + 1):
    # --- PHASE TRAIN ---
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # --- PHASE VALIDATION ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            preds = model(batch_X)
            loss = criterion(preds, batch_y)
            val_loss += loss.item()
            
            # Lấy lớp có xác suất dự đoán cao nhất (torch.max)
            _, predicted_classes = torch.max(preds, dim=1)
            correct += (predicted_classes == batch_y).sum().item()
            total += batch_y.size(0)
            
    val_acc = (correct / total) * 100
    
    if epoch % 3 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | "
              f"Val Acc: {val_acc:.2f}%")

--- 🔥 BÀI TẬP NÂNG CAO: PHÂN LOẠI ĐA LỚP & ĐÁNH GIÁ VALIDATION ---
Epoch 01/15 | Train Loss: 1.1104 | Val Loss: 1.0865 | Val Acc: 41.67%
Epoch 03/15 | Train Loss: 1.0607 | Val Loss: 1.0924 | Val Acc: 40.00%
Epoch 06/15 | Train Loss: 0.9982 | Val Loss: 1.0839 | Val Acc: 48.33%
Epoch 09/15 | Train Loss: 0.9218 | Val Loss: 1.1424 | Val Acc: 50.00%
Epoch 12/15 | Train Loss: 0.8260 | Val Loss: 1.1725 | Val Acc: 51.67%
Epoch 15/15 | Train Loss: 0.7391 | Val Loss: 1.2170 | Val Acc: 50.00%
